In [1]:
#Load Project Environment
import Pkg
Pkg.activate(dirname(@__DIR__))
Pkg.instantiate()

  Activating project at `c:\Users\pbb62\Documents\Repositories\CHANCE_C.jl`


In [2]:
#Load Packages
using CSV, DataFrames
using DataStructures
using Agents
using Statistics,StatsBase,Distributions
using CategoricalArrays

include(joinpath(dirname(@__DIR__), "src/CHANCE_C.jl"))
using .CHANCE_C

In [37]:
###Load Input data:
##For flood history input
f_df = DataFrame(CSV.File(joinpath(dirname(@__DIR__), "data", "synth_flood_phil.csv")))

##For BG
#open bg file
phil_bg = DataFrame(CSV.File(joinpath(dirname(@__DIR__), "data/philly_bg_2019.csv")))
#groupby BG
grouped_phil_bg = groupby(phil_bg, :GEOID)

##load pop data
phil_cbsa_base_pop = DataFrame(CSV.File(joinpath(dirname(dirname(@__DIR__)), "philadelphia-data/census_data/synth_pop/pop_files/philly_cbsa_pop_0.csv")))
#drop missing values
dropmissing!(phil_cbsa_base_pop, :NP)
#drop rows with negative income
#subset!(phil_cbsa_base_pop, :adj_income_2019 .=> ByRow(!<(0)))

#Subset to Phil. County (Not part of function)
phil_base_pop = subset(phil_cbsa_base_pop, :county => x -> x .== 42101)

Row,serialno,year,state,puma,rep,county,tract,bg,puma10,GEOID,RAC1P,NP,HINCP,ADJINC,adj_income_2019
,String15,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64?,Float64,Float64?,Float64?,Float64?
1,2015000000403,2015,42,3201,0,42101,35100,1,4203201,421010351001,1.0,3.0,100000.0,1.08047,108047.0
2,2015000000403,2015,42,3201,0,42101,35200,1,4203201,421010352001,1.0,3.0,100000.0,1.08047,108047.0
3,2015000000403,2015,42,3201,0,42101,35500,3,4203201,421010355003,1.0,3.0,100000.0,1.08047,108047.0
4,2015000000403,2015,42,3201,0,42101,35500,3,4203201,421010355003,1.0,3.0,100000.0,1.08047,108047.0
5,2015000000403,2015,42,3201,0,42101,36100,1,4203201,421010361001,1.0,3.0,100000.0,1.08047,108047.0
6,2015000000403,2015,42,3201,0,42101,36201,3,4203201,421010362013,1.0,3.0,100000.0,1.08047,108047.0
7,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0
8,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0
9,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0


In [33]:
phil_bg.counts

3948-element Vector{Int64}:
   1
  79
 578
   4
 302
 750
  56
 214
 637
   0
   ⋮
   0
   0
  10
   0
   0
  61
   4
   0
   0

In [77]:
#Define input Parameters
no_of_years = 38
start_year = 1981
no_hhs_per_agent=10
growth_rate = 0.01
grouped = true
group_col = "adj_income_2019"
cutoff_dict = OrderedDict(1 => [-60000.00,25000.00], 2 =>[25000.00,75000.00], 3 =>[75000.00, 1e7]) #1=> "low income", 2=> "medium income", 3=> "high income"
bg_cat = Dict(:col =>"income_cat", :group => [1,2,3])
house_budget_mode = "perc"
house_choice_mode = "flood_mem_utility"
risk_averse = 0.3
flood_mem = 10
seed = 1500

1500

In [78]:
### Calculate Flood matrix and Dict for ABM input
f_matrix, f_dict = CHANCE_C.flood_history(f_df; no_of_years = no_of_years, start_year = start_year)

([0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;;], Dict(5 => (1, 5), 16 => (1, 16), 20 => (1, 20), 35 => (1, 35), 12 => (1, 12), 24 => (1, 24), 28 => (1, 28), 8 => (1, 8), 17 => (1, 17), 30 => (1, 30)…))

In [ ]:
f_dict[2]

In [129]:
### Initialize ABM
phil_abm = CHANCE_C.Simulator(phil_bg, phil_base_pop, f_matrix, f_dict, CHANCE_C.model_step!; no_of_years = no_of_years, no_hhs_per_agent = no_hhs_per_agent,
house_budget_mode = house_budget_mode, house_choice_mode = house_choice_mode, grouped = grouped, group_col = group_col, cutoff_dict = cutoff_dict, bg_cat = bg_cat,
risk_averse = risk_averse, flood_mem = flood_mem, seed = seed)

StandardABM with 92967 agents of type Union{BlockGroup, HHAgent, Main.CHANCE_C.Queue}
 agents container: Dict
 space: GridSpace with size (37, 37), metric=chebyshev, periodic=true
 scheduler: Agents.Schedulers.ByType
 properties: df, total_population, flood_hazard, agent_creation, relo_sampler, agent_relocate, build_develop, house_price, hh_utilities_df, no_of_years, flood_matrix, flood_dict, tick

In [ ]:
length([a for a in allagents(phil_abm) if a isa HHAgent])

In [112]:
function AgentMigration(model::ABM; growth_rate = 0.01)
    if growth_rate == 0.0 #In-migration not considered
        return #do nothing
    else
        migrant_ids = [a.id for a in agents_in_position(model[-1].pos, model) if a isa CHANCE_C.HHAgent]
        no_new_agents = floor(Int64, (length([a for a in allagents(model) if a isa CHANCE_C.HHAgent]) - length(migrant_ids)) * growth_rate)
        #Sample from migrant agent pool 
        incoming_ids = sample(abmrng(model), migrant_ids, no_new_agents)
        #move agents to relocation queue
        for id in incoming_ids
            move_agent!(model[id], model[0].pos, model)
        end
    end
end

AgentMigration (generic function with 1 method)

In [130]:
migrant_ids = [a.id for a in agents_in_position(phil_abm[-1].pos, phil_abm) if a isa CHANCE_C.HHAgent]
no_new_agents = floor(Int64, (length([a for a in allagents(phil_abm) if a isa CHANCE_C.HHAgent]) - length(migrant_ids)) * growth_rate)

626

In [131]:
phil_abm.tick += 1
AgentMigration(phil_abm; growth_rate = 0.01)

In [132]:
for id in Agents.schedule(phil_abm)
    if phil_abm[id] isa CHANCE_C.Queue || (phil_abm[id] isa HHAgent && phil_abm[id].bg_id < 1) #Dont involve HHAgents in Queues
        continue
    else
        CHANCE_C.agent_step!(phil_abm[id],phil_abm)
    end
end

In [133]:
collect(agents_in_position(phil_abm[0], phil_abm))

2249-element Vector{AbstractAgent}:
 Main.CHANCE_C.Queue(0, (6, 2), :relocating)
 HHAgent(73527, (6, 2), -1, 10, 1, 8.0, 5, 18566.136000000002, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 24692.960880000006, 0.33)
 HHAgent(71196, (6, 2), -1, 10, 1, 2.0, 1, 24243.480000000007, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 32243.82840000001, 0.33)
 HHAgent(83059, (6, 2), -1, 10, 2, 9.0, 6, 67197.90740000001, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 89373.21684200002, 0.33)
 HHAgent(72024, (6, 2), -1, 10, 1, 2.0, 1, 15819.09, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 21

In [103]:
function Agent_Location(agent::CHANCE_C.Queue, model::ABM; levee = false, f_e = 0.0, bg_sample_size = 10, house_choice_mode = "simple_anova_utility",
    budget_reduction_perc = 0.10, penalty = 50, migrate_prob = 0.05)
    if agent.type == :relocating
        print("correct type")
        loc_df = copy(model.df)
        # Create a GEOID-to-BlockGroup lookup
        geoid_to_bg = Dict{Int64, Int64}()
        for bg in allagents(model)
            if bg isa CHANCE_C.BlockGroup
                geoid_to_bg[bg.GEOID] = bg.id
            end
        end

        # Use view or filter instead of multiple list comprehensions
        moving_agents = sort!([a for a in agents_in_position(agent, model) if a isa CHANCE_C.HHAgent], by=a -> a.income, rev=true)

        current_index = 1
        #Preallocate some vectors to reduce memory allocations
        hh_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
        bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
        bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
        bg_cat = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
        bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

        for hh_agent in moving_agents
            # Consolidate budget selection logic
            bg_budget = if house_choice_mode == "simple_avoidance_utility"
                hh_agent.avoidance ? 
                    subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
                    subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true)
            elseif house_choice_mode == "budget_reduction"
                new_house_budget = hh_agent.house_budget * (1 - budget_reduction_perc)
                hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, hh_agent.house_budget)
                subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
            else
                subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
            end


            # Use a more efficient sampling approach
            try
                # Precompute weights to avoid repeated calculations
                weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
                        
                # Check for available locations more efficiently
                valid_locations = findall(weights .> 0)
                if isempty(valid_locations)
                    throw(ErrorException("No affordable locations with available units"))
                end

                #Sample from affordable locations based on weights
                sample_size = min(length(valid_locations), bg_sample_size)
                sampled_indices = sample(abmrng(model), valid_locations, sample_size, replace=false)
                    
                #Grab utilities from sampled locations
                loc_utilities = [model[geoid_to_bg[row.GEOID]].current_utility[row.income_cat] - ((row.income_cat - hh_agent.group) * penalty) for row in eachrow(bg_budget[sampled_indices, [:GEOID, :income_cat]])]
                # Find indices of block groups with better utilities than current agent location
                current_utility = first(values(hh_agent.utility))
                opt_locs = findall(>(current_utility), loc_utilities)

                # Check if any moves are possible
                if isempty(opt_locs)
                    throw(ErrorException("No better locations found"))
                end
                best_indices = sampled_indices[opt_locs]
                    
                #Append future block group properties to vectors
                ind_length = length(best_indices)

                copyto!(hh_ids, current_index, fill(hh_agent.id, ind_length), 1, ind_length)
                copyto!(bg_ids, current_index, getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID]), 1, ind_length)
                copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
                copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
                copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)

                current_index += ind_length
             
            catch
                # Migration logic remains similar
                last_bg = model[first(keys(hh_agent.utility))]
                if last_bg.id == -1
                    remove_agent!(hh_agent, model)
                    continue
                end
                
                if rand(abmrng(model), Binomial(1, migrate_prob)) == 1
                    move_agent!(hh_agent, last_bg.pos, model)
                    last_bg.occupied_units[hh_agent.group] += 1
                    last_bg.available_units[hh_agent.group] -= 1
                    last_bg.population += getproperty(hh_agent, :no_hhs_per_agent) * getproperty(hh_agent, :hh_size)
                else
                    remove_agent!(hh_agent, model)
                end
            end
            
        end
        
        ##Create df from vectors, append to model properties df
        #Remove extra undef values by using current index
        bg_sample = DataFrame(hh_id = hh_ids[1:current_index-1], bg_id = bg_ids[1:current_index-1], 
        GEOID = bg_GEOID[1:current_index-1], cat = bg_cat[1:current_index-1], bg_utility = bg_utilities[1:current_index-1])
    
        append!(model.hh_utilities_df, bg_sample)
    else
        return
    end
end

Agent_Location (generic function with 1 method)

In [ ]:
function agent_locate(agent::CHANCE_C.Queue, model::ABM; levee = false, f_e = 0.0, bg_sample_size = 10, house_choice_mode = "simple_anova_utility",
    budget_reduction_perc = 0.10, penalty = 50, migrate_prob = 0.05)
    
    if agent.type == :relocating
        loc_df = copy(model.df)
        # Create a GEOID-to-BlockGroup lookup
        geoid_to_bg = Dict{Int64, Int64}()
        for bg in allagents(model)
            if bg isa BlockGroup
                geoid_to_bg[bg.GEOID] = bg.id
            end
        end

        # Use view or filter instead of multiple list comprehensions
        moving_agents = sort!([a for a in agents_in_position(agent, model) if a isa HHAgent], by=a -> a.income, rev=true)

        current_index = 1
        #Preallocate some vectors to reduce memory allocations
        hh_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
        bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
        bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
        bg_cat = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
        bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

        for hh_agent in moving_agents
            # Consolidate budget selection logic
            bg_budget = if house_choice_mode == "simple_avoidance_utility"
                hh_agent.avoidance ? 
                    subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
                    subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true)
            elseif house_choice_mode == "budget_reduction"
                new_house_budget = hh_agent.house_budget * (1 - budget_reduction_perc)
                hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, hh_agent.house_budget)
                subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
            else
                subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
            end


            # Use a more efficient sampling approach
            util_diff = 0
            try
                # Precompute weights to avoid repeated calculations
                weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
                    
                # Check for available locations more efficiently
                valid_locations = findall(weights .> 0)
                if isempty(valid_locations)
                    throw(ErrorException("No affordable locations with available units"))
                end
            
                
                
                #Sample from affordable locations based on weights
                sample_size = min(length(valid_locations), bg_sample_size)
                sampled_indices = sample(abmrng(model), valid_locations, sample_size, replace=false)
                    
                #Grab utilities from sampled locations
                loc_utilities = [model[geoid_to_bg[row.GEOID]].current_utility[row.income_cat] - ((row.income_cat - hh_agent.group) * penalty) for row in eachrow(bg_budget[sampled_indices, [:GEOID, :income_cat]])]
                # Find indices of block groups with better utilities than current agent location
                current_utility = first(values(hh_agent.utility))
                util_diff = (current_utility - maximum(loc_utilities)) / current_utility
                opt_locs = findall(>(current_utility), loc_utilities)

                # Check if any moves are possible
                if isempty(opt_locs)
                    throw(ErrorException("No better locations found"))
                end
                best_indices = sampled_indices[opt_locs]
                    
                #Append future block group properties to vectors
                ind_length = length(best_indices)

                copyto!(hh_ids, current_index, fill(hh_agent.id, ind_length), 1, ind_length)
                copyto!(bg_ids, current_index, getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID]), 1, ind_length)
                copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
                copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
                copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)

                current_index += ind_length
            catch
                # Migration logic remains similar
                last_bg = model[first(keys(hh_agent.utility))]
                if last_bg.id == -1
                    remove_agent!(hh_agent, model)
                    continue
                end
                #Calculate migration probability
                print(util_diff)
                stay_prob = 1/(1+ exp(-3(util_diff)))
                if rand(abmrng(model), Binomial(1, stay_prob)) == 1
                    move_agent!(hh_agent, last_bg.pos, model)
                    last_bg.occupied_units[hh_agent.group] += 1
                    last_bg.available_units[hh_agent.group] -= 1
                    last_bg.population += getproperty(hh_agent, :no_hhs_per_agent) * getproperty(hh_agent, :hh_size)
                else
                    remove_agent!(hh_agent, model)
                end
            end
                
        end
        
        ##Create df from vectors, append to model properties df
        #Remove extra undef values by using current index
        bg_sample = DataFrame(hh_id = hh_ids[1:current_index-1], bg_id = bg_ids[1:current_index-1], 
        GEOID = bg_GEOID[1:current_index-1], cat = bg_cat[1:current_index-1], bg_utility = bg_utilities[1:current_index-1])
        
        append!(model.hh_utilities_df, bg_sample)
    else
        return 
    end
end

agent_locate (generic function with 1 method)

In [135]:
agent_locate(phil_abm[0], phil_abm)

0.119011092498872140.235679720268353070.0112555088740461840.26481255334129430.128507369539349630.0412911865126897940.145641770921111140.092502172608000350.275066297479430070.22670510565212070.075317314722837940.276646161363338360.013144651029745120.135791070007613120.238274018149890330.140790710223037520.233714199103169930.0091823683864711780.226106147529359620.27731038260798790.044345877929628990.135389193141667280.08157243722798510.27923968036536970.25966432135929730.210835914586639170.035898480404668570.0066725365329513980.0258600836794129730.17193430186433580.114793536054631840.167547174903416720.042091367159383420.138962798300344830.330523043613021870.0177620569137740650.12177337884848110.070171110557882970.197562312908168170.17988903004766410.54252911394896360.150203741937439630.112768110222378340.29217167827137540.22093854638064470.0612046430500944450.183851163277398030.049239624151728790.0134781656177410830.23149992648490060.104173861913972150.09221883773659790.3114418728760080

Row,hh_id,bg_id,GEOID,cat,bg_utility
,Int64,Int64,Int64,Int64,Float64
1,89025,151,421010040011,2,3.04929e5
2,89025,31,421010011012,3,3.44807e5
3,89025,241,421010074002,2,3.43606e5
4,89025,1169,421010342002,2,1.47753e5
5,89025,1067,421010316006,3,2.82214e5
6,89025,626,421010188005,1,2.99031e5
7,89025,150,421010039024,3,3.63681e5
8,89025,1226,421010360001,3,1.83992e5
9,89025,749,421010238005,2,4.47732e5


In [127]:
collect(agents_in_position(phil_abm[0], phil_abm))

1282-element Vector{AbstractAgent}:
 Main.CHANCE_C.Queue(0, (6, 2), :relocating)
 HHAgent(73527, (6, 2), -1, 10, 1, 8.0, 5, 18566.136000000002, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 24692.960880000006, 0.33)
 HHAgent(71196, (6, 2), -1, 10, 1, 2.0, 1, 24243.480000000007, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 32243.82840000001, 0.33)
 HHAgent(83059, (6, 2), -1, 10, 2, 9.0, 6, 67197.90740000001, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 89373.21684200002, 0.33)
 HHAgent(72024, (6, 2), -1, 10, 1, 2.0, 1, 15819.09, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 21

In [110]:
phil_abm.hh_utilities_df

Row,hh_id,bg_id,GEOID,cat,bg_utility
,Int64,Int64,Int64,Int64,Float64
1,89025,151,421010040011,2,3.04929e5
2,89025,31,421010011012,3,3.44807e5
3,89025,241,421010074002,2,3.43606e5
4,89025,1169,421010342002,2,1.47753e5
5,89025,1067,421010316006,3,2.82214e5
6,89025,626,421010188005,1,2.99031e5
7,89025,150,421010039024,3,3.63681e5
8,89025,1226,421010360001,3,1.83992e5
9,89025,749,421010238005,2,4.47732e5


In [ ]:
##Extract pop characteristics from pop_df
pop_df = copy(phil_cbsa_base_pop)
group_col = "adj_income_2019"
cutoffs = OrderedDict("low"=> [0,25000.00], "medium"=>[25000.00,75000.00], "high"=>[75000.00, 1e7])
no_hhs_per_agent = 10
#Subset to only occupied households
pop_hh_df = subset(pop_df, :NP => x -> x .> 0.0)

#Create group labels by group col
pop_hh_df[:, :category] = cut(pop_hh_df[:, group_col], unique(reduce(vcat, collect(values(cutoffs)))), labels = collect(keys(cutoffs)))
#groupby category column 
pop_cat_df = groupby(pop_hh_df, :category)

#Create empty DataFrame
agent_df = DataFrame(nrow = Int64[], cat = String[], race = Float64[], avg_hh_size = Float64[], avg_income = Float64[])
for (i,sub_df) in enumerate(pop_cat_df)
    sort!(sub_df, :adj_income_2019)
    sub_df[:,:group] = map(x->div(x,no_hhs_per_agent), 1:nrow(sub_df))
    hh_bins = combine(groupby(sub_df, :group), nrow, :category => maximum => :cat, :RAC1P => (r -> mode(r)) => :race,  [:NP, :adj_income_2019] .=> mean .=> [:avg_hh_size, :avg_income])
    inc_w = ProbabilityWeights(hh_bins.avg_income ./ sum(hh_bins.avg_income)) #Calculate weights based on avg income
    append!(agent_df, hh_bins[sample(abmrng(phil_abm), 1:nrow(hh_bins), inc_w, migrant_cat_pop[i]; replace = true),2:end]) #Sample rows based on migrant category count
end

In [ ]:
agent_df

In [ ]:
t_g = groupby(agent_df, :cat)[1]
inc_w = ProbabilityWeights(t_g.avg_income ./ sum(t_g.avg_income))
t_g[sample(abmrng(phil_abm), 1:nrow(t_g), inc_w, migrant_cat_pop[1]; replace = true), :]

In [ ]:
### Test model functions
test_bg = phil_abm[10]
println("Occupied: ",test_bg.occupied_units)
println("Available: ",test_bg.available_units)
println("Population: ",test_bg.population)

In [ ]:
length([a for a in agents_in_position(test_bg, phil_abm) if a isa HHAgent && a.group == "medium"])

In [ ]:
CHANCE_C.agent_prob!(test_bg, phil_abm)

In [ ]:
println("Occupied: ",test_bg.occupied_units)
println("Available: ",test_bg.available_units)
println("Population: ",test_bg.population)

In [ ]:
collect(agents_in_position(phil_abm[0].pos, phil_abm))

In [ ]:
function agent_locate(agent::CHANCE_C.Queue, model::ABM; levee = false, f_e = 0.0, bg_sample_size = 10, house_choice_mode = "simple_anova_utility",
    budget_reduction_perc = 0.10, penalty = 50, migrate_prob = 0.05)
    
    loc_df = copy(model.df)
    # Preallocate the DataFrame with a reasonable initial capacity
    #bg_sample = DataFrame(hh_id = Int64[], bg_id = Int64[], GEOID = Int64[], cat = String[], bg_utility = Float64[])

    # Use view or filter instead of multiple list comprehensions
    moving_agents = sort!([a for a in agents_in_position(agent, model) if a isa HHAgent], by=a -> a.income, rev=true)

    current_index = 1
    #Preallocate some vectors to reduce memory allocations
    hh_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
    bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
    bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

    for hh_agent in moving_agents
        # Consolidate budget selection logic
        bg_budget = if house_choice_mode == "simple_avoidance_utility"
            hh_agent.avoidance ? 
                subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
                subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true)
        elseif house_choice_mode == "budget_reduction"
            new_house_budget = hh_agent.house_budget * (1 - budget_reduction_perc)
            hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, hh_agent.house_budget)
            subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
        else
            subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
        end

        # Use a more efficient sampling approach
        try
            
            # Precompute weights to avoid repeated calculations
            weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
            
            # Check for available locations more efficiently
            valid_locations = findall(weights .> 0)
            if isempty(valid_locations)
                throw(ErrorException("No affordable locations with available units"))
            end

            #Sample from affordable locations based on weights
            sample_size = min(length(valid_locations), bg_sample_size)
            sampled_indices = sample(abmrng(model), valid_locations, sample_size, replace=false)
    
            #Grab utilities from sampled locations
            bg_sel = Iterators.filter(bg -> bg isa BlockGroup && bg.GEOID in bg_budget[sampled_indices, :GEOID], allagents(model)).id
            loc_utilities = getindex.(getproperty.(getindex.(Ref(model), bg_sel), :current_utility), bg_budget[sampled_indices, :income_cat])
            # Find indices of block groups with better utilities than current agent location
            current_utility = first(values(hh_agent.utility))
            opt_locs = findall(>(current_utility), loc_utilities)

            # Check if any moves are possible
            if isempty(opt_locs)
                throw(ErrorException("No better locations found"))
            end
            best_indices = sampled_indices[opt_locs]

            #Append future block group properties to vectors
            ind_length = length(best_indices)

            copyto!(hh_ids, current_index, fill(hh_agent.id, ind_length), 1, ind_length)
            copyto!(bg_ids, current_index, collect(bg_sel)[opt_locs], 1, ind_length)
            copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
            copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
            copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)

            current_index += ind_length

        catch
            # Migration logic remains similar
            if rand(abmrng(model), Binomial(1, migrate_prob)) == 1
                last_bg = model[first(keys(hh_agent.utility))]
                move_agent!(hh_agent, last_bg.pos, model)
                last_bg.occupied_units[hh_agent.group] += 1
                last_bg.available_units[hh_agent.group] -= 1
                last_bg.population += getproperty(hh_agent, :no_hhs_per_agent) * getproperty(hh_agent, :hh_size)
            else
                remove_agent!(hh_agent, model)
            end
        end
    end
    
    ##Create df from vectors, append to model properties df
    #Remove extra undef values by using current index
    bg_sample = DataFrame(hh_id = hh_ids[1:current_index-1], bg_id = bg_ids[1:current_index-1], 
    GEOID = bg_GEOID[1:current_index-1], cat = bg_cat[1:current_index-1], bg_utility = bg_utilities[1:current_index-1])
    
    append!(model.hh_utilities_df, bg_sample)
end

In [ ]:
agent_locate(phil_abm[0], phil_abm)


In [ ]:
phil_abm.hh_utilities_df

In [ ]:
#Breakdown agent relocation function:
loc_df = copy(phil_abm.df)
#Create a GEOID-to-BlockGroup lookup
geoid_to_bg = Dict{Int64, Int64}()
for bg in allagents(phil_abm)
    if bg isa BlockGroup
        geoid_to_bg[bg.GEOID] = bg.id
    end
end
house_choice_mode = "simple_anova_utility"
bg_sample_size = 10
# Use view or filter instead of multiple list comprehensions
moving_agents = sort!([a for a in agents_in_position(phil_abm[0], phil_abm) if a isa CHANCE_C.HHAgent], by=a -> a.income, rev=true)
agent_sel = moving_agents[1]
current_index = 1
# Preallocate some vectors to reduce memory allocations
bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

# Consolidate budget selection logic
bg_budget = if house_choice_mode == "simple_avoidance_utility"
    agent_sel.avoidance ? 
        subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
        subset(loc_df, :market_value => n -> n .<= agent_sel.house_budget, skipmissing=true)
elseif house_choice_mode == "budget_reduction"
    new_house_budget = agent_sel.house_budget * (1 - budget_reduction_perc)
    hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, agent_sel.house_budget)
    subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
else
    subset(loc_df, :market_value => n -> n .<= agent_sel.house_budget, skipmissing=true, view = true)
end




In [ ]:
# Use a more efficient sampling approach
#try
    # Precompute weights to avoid repeated calculations
weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
            
    # Check for available locations more efficiently
valid_locations = findall(weights .> 0)
if isempty(valid_locations)
    throw(ErrorException("No affordable locations with available units"))
end
    # Efficient sampling
sample_size = min(length(valid_locations), bg_sample_size)
sampled_indices = sample(abmrng(phil_abm), valid_locations, sample_size, replace=false)
    #bg_options = bg_budget[sampled_indices, :]
    
    #Grab utilities from sampled locations
#bg_sel = map(bg -> bg.id, Iterators.filter(bg -> bg isa BlockGroup && bg.GEOID in bg_budget[sampled_indices, :GEOID], allagents(phil_abm)))
loc_utilities = [phil_abm[geoid_to_bg[row.GEOID]].current_utility[row.income_cat] for row in eachrow(bg_budget[sampled_indices, [:GEOID, :income_cat]])]
# Find indices of block groups with better utilities than current agent location
current_utility = first(values(agent_sel.utility))
opt_locs = findall(>(current_utility), loc_utilities)
# Check if any moves are possible
if isempty(opt_locs)
    throw(ErrorException("No better locations found"))
end
best_indices = sampled_indices[opt_locs]
#catch
#    println("didnt work!")
#end

In [ ]:
println(sampled_indices)
println(opt_locs)
println(best_indices)


In [ ]:
getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID])

In [ ]:
ind_length = length(best_indices)   
#bg_ids = getproperty.(bg_sel[opt_locs], :id)
#bg_GEOID = bg_budget[sampled_indices, :GEOID]
#bg_cat = bg_budget[sampled_indices, :income_cat]
#bg_utilities = loc_utilities[opt_locs]

copyto!(bg_ids, current_index, getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID]), 1, ind_length)
copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)
current_index += ind_length


# Append to bg_sample
#append!(bg_sample, move_df)

In [ ]:
fill(agent_sel.id, ind_length)

In [ ]:
agent_locate(phil_abm[0], phil_abm)

In [ ]:
agent_relocate(phil_abm[0], phil_abm)

In [ ]:
sort!(filter(a -> a isa HHAgent, agents_in_position(phil_abm[0], phil_abm)), by=a -> a.income, rev=true)

In [ ]:
step!(phil_abm)

In [ ]:
phil_abm.tick